# Statistical Exchange of Space and Time

Computer algorithms tend to solve the same problem over a spectrum of space-time trade-offs. 
It's reasonable to expect the same from statistical estimation

In [1]:
## Dense net code initially authored by Google's search engine GenAI on 20 Oct 2024. 
## I've applied minor modifications for generality, but the code worked great on first draft. 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

## define control 
class DenseNetControl(nn.Module):
    def __init__(self):
        super(DenseNetControl, self).__init__()
        self.features = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        if x.shape[0] == 0: 
            return torch.tensor([])
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return x
    pass 

## define experimental model 
class DenseNetExperimental(nn.Module):
    def __init__(self):
        super(DenseNetExperimental, self).__init__() 
        self.linear = nn.Linear(128, 128)
        self.features = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        if x.shape[0] == 0: 
            return torch.tensor([])
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return x
    pass 

## Load MNIST dataset
train_dataset = datasets.MNIST(root='/tmp/data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='/tmp/data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [2]:
## Dense net experiment, needs about 1-2 min of compute 

# Initialize the model, loss function and optimizer 
model_control = DenseNetControl() 
criterion = nn.CrossEntropyLoss() 
model_control.optimizer = optim.Adam(model_control.parameters(), lr=0.001) 

model_experimental = DenseNetExperimental() 
criterion = nn.CrossEntropyLoss() 
model_experimental.optimizer = optim.Adam(model_experimental.parameters(), lr=0.001) 

# Train the model 
def fit(model):
    model.train() 
    num_epochs = 5 
    for epoch in range(num_epochs): 
        for i, (data, target) in enumerate(train_loader): 
            model.optimizer.zero_grad() 
            output = model(data) 
            loss = criterion(output, target) 
            loss.backward() 
            model.optimizer.step() 
            if i % 100 == 0:
                print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, i+1, len(train_loader), loss.item()))
    # Evaluate the model
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    acc = correct / total
    print('Accuracy of the network on the 10000 test images: {} %'.format(100 * correct / total)) 
    return acc 

In [3]:
acc = fit(model_control)
print(acc)

Epoch [1/5], Step [1/938], Loss: 2.2990
Epoch [1/5], Step [101/938], Loss: 2.3013
Epoch [1/5], Step [201/938], Loss: 2.3092
Epoch [1/5], Step [301/938], Loss: 2.3150
Epoch [1/5], Step [401/938], Loss: 2.3140
Epoch [1/5], Step [501/938], Loss: 2.2997


KeyboardInterrupt: 

In [ ]:
acc = fit(model_experimental)
print(acc)